In [1]:
%load_ext autoreload
%autoreload 2

from utils import get_model, get_data, find_multitoken_words, get_attention, compute_first_vs_last_attn
import torch


/opt/anaconda3/envs/dill/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model, tokenizer = get_model("allenai/Olmo-3-1025-7B")

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1
Fetching 3 files:   0%|          | 0/3 [00:05<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# Grab a sample document
ds = get_data()
sample = next(iter(ds["train"]))
text = sample["text"][:2000]  # truncate to keep attention manageable
print(text[:500], "...")

In [ ]:
# Run forward pass and get attentions
device = "cuda" if torch.cuda.is_available() else "cpu"
attentions, input_ids = get_attention(model, tokenizer, text, max_length=512, device=device)

print(f"Layers: {len(attentions)}, Heads: {attentions[0].shape[1]}, Seq len: {attentions[0].shape[-1]}")

In [ ]:
# Find multi-token words
spans = find_multitoken_words(tokenizer, input_ids)
print(f"Found {len(spans)} multi-token words")

# Show a few examples
for s, e in spans[:10]:
    word_tokens = [tokenizer.decode(input_ids[i]) for i in range(s, e + 1)]
    print(f"  positions {s}-{e}: {word_tokens!r} -> '{''.join(word_tokens).strip()}'")

In [ ]:
# Compute max attention to first vs last subword token
first_max, last_max = compute_first_vs_last_attn(attentions, spans)

# first_max, last_max: (num_layers, num_heads, num_words)
# Average over all multi-token words
first_avg = first_max.mean(dim=-1)  # (layers, heads)
last_avg = last_max.mean(dim=-1)    # (layers, heads)

# Fraction of (layer, head) pairs where last > first
last_wins = (last_avg > first_avg).float()
print(f"Fraction of (layer, head) pairs where last-token attn > first-token attn: "
      f"{last_wins.mean().item():.2%}")
print(f"Mean max-attn to FIRST subword: {first_avg.mean().item():.4f}")
print(f"Mean max-attn to LAST  subword: {last_avg.mean().item():.4f}")

In [ ]:
# Per-layer summary: what fraction of heads prefer last token?
num_layers = first_avg.shape[0]
print(f"{'Layer':>6}  {'Heads last>first':>16}  {'Avg first':>10}  {'Avg last':>10}  {'Ratio':>8}")
print("-" * 60)
for layer in range(num_layers):
    frac = (last_avg[layer] > first_avg[layer]).float().mean().item()
    f_val = first_avg[layer].mean().item()
    l_val = last_avg[layer].mean().item()
    ratio = l_val / f_val if f_val > 0 else float('inf')
    print(f"{layer:>6}  {frac:>16.0%}  {f_val:>10.4f}  {l_val:>10.4f}  {ratio:>8.2f}x")




    

In [ ]:
import torch
from utils import get_model, get_data_sample, get_multi_token_words

# Load model and get a sample
model, tokenizer = get_model()
text = get_data_sample()
paragraph = text.split("\n\n")[0]

# Step 1: ID multi-token words
multi = get_multi_token_words(paragraph, tokenizer)
print(multi)

# Run the model to get attention weights
inputs = tokenizer(paragraph, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)
attentions = outputs.attentions

# Steps 2 & 3: For each multi-token word, for each layer & head
for word, info in multi.items():
    first_pos = info["positions"][0]
    last_pos = info["positions"][-1]
    
    print(f"\n=== {word} ({info['tokens']}) ===")
    
    for layer in range(len(attentions)):
        for head in range(attentions[layer].shape[1]):
            # Step 2: max attn from FIRST subtoken to tokens after last subtoken
            step2 = attentions[layer][0, head, first_pos, last_pos + 1:].max().item()
            
            # Step 3: max attn from LAST subtoken to tokens after last subtoken
            step3 = attentions[layer][0, head, last_pos, last_pos + 1:].max().item()
            
            print(f"  Layer {layer:2d}, Head {head:2d}: first->{step2:.4f}, last->{step3:.4f}")